# 01 — Vehicle Detection on LVO Satellite Imagery (GeoTIFF + Vision Verification)

**Objective:** Count military vehicles at Russian LVO garrisons using full-resolution GeoTIFF satellite imagery.

**Pipeline:**
1. Load GeoTIFF (16-bit multispectral) → normalize to 8-bit RGB
2. Tile into 640×640 patches → YOLOv8x detection
3. Global NMS deduplication across tiles
4. **Vision model verification** — each detection cropped and sent to Qwen2.5-VL to classify: military vehicle / civilian vehicle / false positive
5. Only verified detections kept in final count

**Data:** [Estwarden/dataset](https://github.com/Estwarden/dataset) | **Sources:** [ISW](https://www.understandingwar.org/), [SkyFi](https://skyfi.com)

In [ ]:
import os, json, base64, requests
import numpy as np
import cv2
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
from PIL import Image as PILImage
PILImage.MAX_IMAGE_PIXELS = None
from pathlib import Path
from ultralytics import YOLO
from collections import Counter
from io import BytesIO

GEOTIFF_DIR = Path("../data/geotiff")
OUTPUT_DIR = Path("../outputs/01-vehicle-detection")
FINDINGS_DIR = OUTPUT_DIR / "findings"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINDINGS_DIR.mkdir(parents=True, exist_ok=True)

VISION_URL = "http://localhost:11434/api/chat"  # Ollama Qwen2.5-VL
VISION_MODEL = "qwen2.5vl:7b"

PSKOV_TIF = list(GEOTIFF_DIR.glob("pskov-76vdv/*.tif"))[0]
CHEREKHA_TIF = list(GEOTIFF_DIR.glob("pskov-cherekha/*.tif"))[0]

for tif in [PSKOV_TIF, CHEREKHA_TIF]:
    with rasterio.open(tif) as ds:
        print(f"{tif.parent.name:20} {ds.width}x{ds.height} bands={ds.count} res={ds.res[0]:.2f}m/px CRS={ds.crs}")

## Step 1: Load GeoTIFF and convert to 8-bit RGB

In [ ]:
def load_geotiff_rgb(path, max_dim=None):
    """Load GeoTIFF, select RGB bands, normalize 16-bit to 8-bit."""
    with rasterio.open(path) as ds:
        # Read first 3 bands (R, G, B) or bands 5,3,2 for WV3
        if ds.count >= 8:  # WV3 multispectral
            # WV3 band order: Coastal, Blue, Green, Yellow, Red, RedEdge, NIR1, NIR2
            r = ds.read(5).astype(float)  # Red
            g = ds.read(3).astype(float)  # Green
            b = ds.read(2).astype(float)  # Blue
            print(f"  WV3 8-band: using bands 5(R), 3(G), 2(B)")
        else:  # SkySat RGBA
            r = ds.read(1).astype(float)
            g = ds.read(2).astype(float)
            b = ds.read(3).astype(float)
            print(f"  SkySat 4-band: using bands 1(R), 2(G), 3(B)")
        
        # Percentile stretch for better contrast
        def stretch(band):
            mask = band > 0  # ignore nodata
            if mask.sum() == 0:
                return np.zeros_like(band, dtype=np.uint8)
            p2 = np.percentile(band[mask], 2)
            p98 = np.percentile(band[mask], 98)
            clipped = np.clip((band - p2) / (p98 - p2 + 1e-6) * 255, 0, 255)
            return clipped.astype(np.uint8)
        
        rgb = np.stack([stretch(r), stretch(g), stretch(b)], axis=-1)
        
        # Create nodata mask (where all bands are 0)
        nodata_mask = (r == 0) & (g == 0) & (b == 0)
        rgb[nodata_mask] = 0
        
        meta = {
            "width": ds.width, "height": ds.height,
            "crs": str(ds.crs), "res": ds.res,
            "bounds": ds.bounds, "bands": ds.count
        }
        print(f"  Output: {rgb.shape[1]}x{rgb.shape[0]} RGB uint8")
        return rgb, meta

print("Loading Pskov SkySat GeoTIFF...")
pskov_rgb, pskov_meta = load_geotiff_rgb(PSKOV_TIF)
print(f"\nLoading Pskov Cherekha WV3 GeoTIFF...")
cherekha_rgb, cherekha_meta = load_geotiff_rgb(CHEREKHA_TIF)

## Step 2: Tile and run YOLOv8x

In [ ]:
def tile_image_array(img, tile_size=640, overlap=64):
    h, w = img.shape[:2]
    stride = tile_size - overlap
    tiles = []
    for y in range(0, h - tile_size + 1, stride):
        for x in range(0, w - tile_size + 1, stride):
            tile = img[y:y+tile_size, x:x+tile_size]
            # Skip tiles that are mostly nodata (black)
            if tile.mean() < 5:
                continue
            tiles.append({"image": tile, "x": x, "y": y})
    print(f"  {w}x{h} → {len(tiles)} valid tiles (skipped nodata)")
    return tiles

model = YOLO("yolov8x.pt")
VEHICLE_CLASSES = {2: "car", 5: "bus", 7: "truck"}

def detect_all(img, name):
    tiles = tile_image_array(img)
    dets = []
    for i, t in enumerate(tiles):
        if i % 200 == 0: print(f"    tile {i}/{len(tiles)}...")
        for r in model.predict(t["image"], conf=0.25, verbose=False, device="cuda"):
            for box in r.boxes:
                cid = int(box.cls[0])
                if cid in VEHICLE_CLASSES:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    dets.append({
                        "class": VEHICLE_CLASSES[cid], "conf": float(box.conf[0]),
                        "x1": int(x1+t["x"]), "y1": int(y1+t["y"]),
                        "x2": int(x2+t["x"]), "y2": int(y2+t["y"]),
                    })
    print(f"  {name}: {len(dets)} raw detections")
    return dets

print("Detecting on Pskov SkySat GeoTIFF...")
det_pskov_raw = detect_all(pskov_rgb, "pskov")
print("\nDetecting on Cherekha WV3 GeoTIFF...")
det_cherekha_raw = detect_all(cherekha_rgb, "cherekha")

## Step 3: Global NMS

In [ ]:
def global_nms(dets, iou_thresh=0.5):
    if not dets: return []
    boxes = np.array([[d["x1"],d["y1"],d["x2"],d["y2"]] for d in dets], dtype=float)
    scores = np.array([d["conf"] for d in dets])
    order = scores.argsort()[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]; keep.append(i)
        if len(order) == 1: break
        xx1 = np.maximum(boxes[i,0], boxes[order[1:],0])
        yy1 = np.maximum(boxes[i,1], boxes[order[1:],1])
        xx2 = np.minimum(boxes[i,2], boxes[order[1:],2])
        yy2 = np.minimum(boxes[i,3], boxes[order[1:],3])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        a_i = (boxes[i,2]-boxes[i,0])*(boxes[i,3]-boxes[i,1])
        a_j = (boxes[order[1:],2]-boxes[order[1:],0])*(boxes[order[1:],3]-boxes[order[1:],1])
        iou = inter / (a_i + a_j - inter + 1e-6)
        order = order[np.where(iou <= iou_thresh)[0] + 1]
    return [dets[i] for i in keep]

det_pskov = global_nms(det_pskov_raw)
det_cherekha = global_nms(det_cherekha_raw)
print(f"Pskov: {len(det_pskov_raw)} raw → {len(det_pskov)} after NMS")
print(f"Cherekha: {len(det_cherekha_raw)} raw → {len(det_cherekha)} after NMS")

## Step 4: Vision Model Verification

Every YOLO detection is cropped (with 2× context padding), encoded as JPEG, and sent to **Qwen2.5-VL 7B** running locally on Ollama.

The vision model classifies each crop as:
- `MILITARY` — tank, APC, IFV, military truck, artillery, SAM
- `CIVILIAN` — car, bus, civilian truck, construction equipment  
- `FALSE_POSITIVE` — building, shadow, road marking, vegetation, artifact

This eliminates YOLO's false positives on satellite imagery where buildings/shadows routinely get classified as vehicles.

In [ ]:
def crop_detection(img, det, padding_factor=2.0):
    """Crop detection with context padding for vision model."""
    x1, y1, x2, y2 = det["x1"], det["y1"], det["x2"], det["y2"]
    w, h = x2 - x1, y2 - y1
    pw, ph = int(w * padding_factor), int(h * padding_factor)
    cx1 = max(0, x1 - pw)
    cy1 = max(0, y1 - ph)
    cx2 = min(img.shape[1], x2 + pw)
    cy2 = min(img.shape[0], y2 + ph)
    return img[cy1:cy2, cx1:cx2]

def verify_with_vision(crop_rgb, det_class, det_conf):
    """Send crop to Qwen2.5-VL for classification."""
    # Resize to max 512px for speed
    h, w = crop_rgb.shape[:2]
    scale = min(512/w, 512/h, 1.0)
    if scale < 1:
        crop_rgb = cv2.resize(crop_rgb, (int(w*scale), int(h*scale)))
    
    # Encode as JPEG base64
    _, buf = cv2.imencode(".jpg", cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 90])
    b64 = base64.b64encode(buf).decode()
    
    prompt = (
        "This is a satellite image crop at 50cm or 35cm resolution. "
        "An object detector flagged this area as containing a vehicle. "
        "Classify what you see as exactly ONE of: MILITARY, CIVILIAN, or FALSE_POSITIVE.\n\n"
        "MILITARY = tank, APC, IFV, military truck, artillery, SAM system, military helicopter\n"
        "CIVILIAN = passenger car, civilian truck, bus, construction vehicle\n"  
        "FALSE_POSITIVE = building roof, shadow, road marking, vegetation, image artifact, nothing\n\n"
        "Respond with ONLY the classification word and a brief reason. "
        "Example: 'FALSE_POSITIVE — building shadow'"
    )
    
    try:
        resp = requests.post(VISION_URL, json={
            "model": VISION_MODEL,
            "messages": [{"role": "user", "content": prompt, "images": [b64]}],
            "stream": False
        }, timeout=30)
        answer = resp.json().get("message", {}).get("content", "UNKNOWN")
        
        # Parse classification
        for label in ["MILITARY", "CIVILIAN", "FALSE_POSITIVE"]:
            if label in answer.upper():
                return label, answer.strip()
        return "UNKNOWN", answer.strip()
    except Exception as e:
        return "ERROR", str(e)

# Verify all detections
print(f"Verifying {len(det_pskov)} Pskov + {len(det_cherekha)} Cherekha detections with {VISION_MODEL}...")
print()

for site_name, dets, img in [("pskov", det_pskov, pskov_rgb), ("cherekha", det_cherekha, cherekha_rgb)]:
    print(f"--- {site_name}: {len(dets)} detections ---")
    for i, det in enumerate(dets):
        crop = crop_detection(img, det)
        label, reason = verify_with_vision(crop, det["class"], det["conf"])
        det["verified"] = label
        det["vision_reason"] = reason
        
        # Save crop
        crop_path = FINDINGS_DIR / f"{site_name}-det-{i:03d}-{label.lower()}.jpg"
        cv2.imwrite(str(crop_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 92])
        
        if i < 10 or label == "MILITARY":
            print(f"  [{i:3d}] {det['class']:6} conf={det['conf']:.2f} → {label:15} | {reason[:60]}")
    
    print(f"  ... {len(dets)} total verified")
    counts = Counter(d.get("verified","?") for d in dets)
    print(f"  Results: {dict(counts)}")
    print()

## Step 5: Generate annotated output images

In [ ]:
def plot_verified(img, dets, title, out_path, res_m, max_dim=4096):
    """Plot detections colored by verification status."""
    h, w = img.shape[:2]
    scale = min(max_dim/w, max_dim/h, 1.0)
    display = cv2.resize(img, (int(w*scale), int(h*scale))) if scale < 1 else img.copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(20, 16))
    ax.imshow(display)
    
    colors = {"MILITARY": "#ff0000", "CIVILIAN": "#00ff00", "FALSE_POSITIVE": "#666666", "UNKNOWN": "#ffff00"}
    
    military = [d for d in dets if d.get("verified") == "MILITARY"]
    civilian = [d for d in dets if d.get("verified") == "CIVILIAN"]
    fp = [d for d in dets if d.get("verified") == "FALSE_POSITIVE"]
    
    for d in dets:
        color = colors.get(d.get("verified", "UNKNOWN"), "#ffff00")
        x1, y1 = int(d["x1"]*scale), int(d["y1"]*scale)
        x2, y2 = int(d["x2"]*scale), int(d["y2"]*scale)
        lw = 3 if d.get("verified") == "MILITARY" else 1
        ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, lw=lw, ec=color, fc="none"))
    
    legend = f"🔴 Military: {len(military)}  🟢 Civilian: {len(civilian)}  ⚪ False positive: {len(fp)}"
    ax.set_title(f"{title}\n{legend}", fontsize=14)
    ax.axis("off")
    
    # Scale bar
    bar_px = int(200 / res_m * scale)  # 200m
    dh = display.shape[0]
    ax.plot([20, 20+bar_px], [dh-30, dh-30], color="white", linewidth=3)
    ax.text(20, dh-40, "200m", color="white", fontsize=10)
    
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

plot_verified(pskov_rgb, det_pskov, 
              f"Pskov 76th VDV — SkySat 50cm GeoTIFF — 2026-03-14\n{pskov_meta['width']}x{pskov_meta['height']}px, {pskov_meta['bands']} bands, EPSG:32635",
              OUTPUT_DIR / "pskov_76vdv_verified.png", res_m=0.5)

plot_verified(cherekha_rgb, det_cherekha,
              f"Pskov Cherekha — WV3 35cm GeoTIFF — 2026-02-05\n{cherekha_meta['width']}x{cherekha_meta['height']}px, {cherekha_meta['bands']} bands, EPSG:32635",
              OUTPUT_DIR / "pskov_cherekha_verified.png", res_m=0.34)

## Step 6: Key area closeups with annotations

In [ ]:
def save_annotated_crop(img, dets, region, name, title, res_m):
    """Crop a region, draw verified detections, add scale bar."""
    x1, y1, x2, y2 = region
    h, w = img.shape[:2]
    x2, y2 = min(x2, w), min(y2, h)
    crop = img[y1:y2, x1:x2].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(14, 14))
    ax.imshow(crop)
    
    colors = {"MILITARY": "#ff0000", "CIVILIAN": "#00ff00", "FALSE_POSITIVE": "#666666"}
    
    region_dets = [d for d in dets if d["x1"] >= x1 and d["y1"] >= y1 and d["x2"] <= x2 and d["y2"] <= y2]
    
    for d in region_dets:
        color = colors.get(d.get("verified", ""), "#ffff00")
        rx1, ry1 = d["x1"]-x1, d["y1"]-y1
        rx2, ry2 = d["x2"]-x1, d["y2"]-y1
        lw = 3 if d.get("verified") == "MILITARY" else 1
        ax.add_patch(Rectangle((rx1,ry1), rx2-rx1, ry2-ry1, lw=lw, ec=color, fc="none"))
        if d.get("verified") == "MILITARY":
            ax.text(rx1, ry1-5, "MILITARY", color="red", fontsize=8, fontweight="bold",
                   bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.7))
    
    # Scale bar: 100m
    bar_px = int(100 / res_m)
    ch = crop.shape[0]
    ax.plot([10, 10+bar_px], [ch-20, ch-20], color="white", linewidth=3)
    ax.text(10, ch-30, "100m", color="white", fontsize=10)
    
    mil = len([d for d in region_dets if d.get("verified")=="MILITARY"])
    civ = len([d for d in region_dets if d.get("verified")=="CIVILIAN"])
    fp = len([d for d in region_dets if d.get("verified")=="FALSE_POSITIVE"])
    
    ax.set_title(f"{title}\n🔴 Military: {mil}  🟢 Civilian: {civ}  ⚪ FP: {fp}", fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    
    out = FINDINGS_DIR / f"{name}.jpg"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  {name}: {mil} military, {civ} civilian, {fp} FP")
    return mil, civ, fp

# Pskov SkySat key areas
h, w = pskov_rgb.shape[:2]
pskov_crops = {
    "pskov-airfield": (w//3, 0, w, h//2, "Pskov — Airfield & Runway Area"),
    "pskov-garrison": (0, 0, w//3, h//3, "Pskov — Garrison & Barracks"),
    "pskov-vehicle-parks": (0, h//4, w//3, 2*h//3, "Pskov — Vehicle Parks Area"),
    "pskov-south": (0, 2*h//3, w//2, h, "Pskov — Southern Sector"),
}
print("=== Pskov SkySat crops ===")
for name, (*region, title) in pskov_crops.items():
    save_annotated_crop(pskov_rgb, det_pskov, region, name, title, 0.5)

# Cherekha WV3 key areas
h2, w2 = cherekha_rgb.shape[:2]
cherekha_crops = {
    "cherekha-north": (0, 0, w2, h2//3, "Cherekha — Northern Sector"),
    "cherekha-center": (0, h2//3, w2, 2*h2//3, "Cherekha — Central Area"),
    "cherekha-south": (0, 2*h2//3, w2, h2, "Cherekha — Southern Sector"),
}
print("\n=== Cherekha WV3 crops ===")
for name, (*region, title) in cherekha_crops.items():
    save_annotated_crop(cherekha_rgb, det_cherekha, region, name, title, 0.34)

## Step 7: Summary

In [ ]:
mil_p = len([d for d in det_pskov if d.get("verified")=="MILITARY"])
civ_p = len([d for d in det_pskov if d.get("verified")=="CIVILIAN"])
fp_p = len([d for d in det_pskov if d.get("verified")=="FALSE_POSITIVE"])

mil_c = len([d for d in det_cherekha if d.get("verified")=="MILITARY"])
civ_c = len([d for d in det_cherekha if d.get("verified")=="CIVILIAN"])
fp_c = len([d for d in det_cherekha if d.get("verified")=="FALSE_POSITIVE"])

results = {
    "pskov_76vdv": {
        "sensor": "Planet SkySat", "resolution_m": 0.5, "date": "2026-03-14",
        "geotiff_size": f"{pskov_meta['width']}x{pskov_meta['height']}", "bands": pskov_meta["bands"],
        "crs": pskov_meta["crs"],
        "raw_detections": len(det_pskov_raw), "after_nms": len(det_pskov),
        "verified_military": mil_p, "verified_civilian": civ_p, "false_positive": fp_p,
    },
    "pskov_cherekha": {
        "sensor": "Maxar WorldView-3", "resolution_m": 0.34, "date": "2026-02-05",
        "geotiff_size": f"{cherekha_meta['width']}x{cherekha_meta['height']}", "bands": cherekha_meta["bands"],
        "crs": cherekha_meta["crs"],
        "raw_detections": len(det_cherekha_raw), "after_nms": len(det_cherekha),
        "verified_military": mil_c, "verified_civilian": civ_c, "false_positive": fp_c,
    }
}

print("=" * 70)
print("LVO FORCE POSTURE — VERIFIED VEHICLE DETECTION")
print("=" * 70)
for n, r in results.items():
    print(f"\n  {n}:")
    print(f"    Source: {r['sensor']} {r['resolution_m']}m, {r['date']}")
    print(f"    GeoTIFF: {r['geotiff_size']}, {r['bands']} bands, {r['crs']}")
    print(f"    Pipeline: {r['raw_detections']} raw → {r['after_nms']} NMS → vision verified")
    print(f"    🔴 MILITARY:       {r['verified_military']}")
    print(f"    🟢 CIVILIAN:       {r['verified_civilian']}")
    print(f"    ⚪ FALSE POSITIVE: {r['false_positive']}")

total_mil = mil_p + mil_c
total_civ = civ_p + civ_c
total_fp = fp_p + fp_c
print(f"\n{'=' * 70}")
print(f"  TOTAL MILITARY VEHICLES: {total_mil}")
print(f"  TOTAL CIVILIAN:          {total_civ}")
print(f"  FALSE POSITIVES REMOVED: {total_fp}")
print(f"{'=' * 70}")

with open(OUTPUT_DIR / "results_verified.json", "w") as f:
    json.dump(results, f, indent=2)

# Save all detections with verification
with open(OUTPUT_DIR / "detections_pskov.json", "w") as f:
    json.dump(det_pskov, f, indent=2)
with open(OUTPUT_DIR / "detections_cherekha.json", "w") as f:
    json.dump(det_cherekha, f, indent=2)
    
print(f"\nSaved to {OUTPUT_DIR}")

## Methodology Notes

- **GeoTIFF source data**: 16-bit multispectral, georeferenced (EPSG:32635/UTM 35N)
- **SkySat**: 4 bands (R,G,B,A) at 0.50m/px — each pixel = 50cm ground
- **WorldView-3**: 8 bands (multispectral) at 0.34m/px — each pixel = 34cm ground
- **Normalization**: 2nd–98th percentile stretch per band
- **Detection**: YOLOv8x (COCO pretrained), conf threshold 0.25
- **Verification**: Every detection reviewed by [Qwen2.5-VL 7B](https://huggingface.co/Qwen/Qwen2.5-VL-7B) vision model
- **False positive sources**: building roofs, shadows, road markings, vegetation edges

All data: [Estwarden/dataset](https://github.com/Estwarden/dataset)
ISW unit tracking: [source](https://www.understandingwar.org/backgrounder/russian-offensive-campaign-assessment)